# Projet SM604 — Mathématiques pour le Machine Learning
## De la classification des chiffres manuscrits à la détection de cancers du sein
### EFREI Paris — Année 2025-2026 / Semestre 6

---

**Objectif :** Implémenter des méthodes d'apprentissage supervisé (modèle linéaire, MLP, CNN) pour résoudre des problèmes de classification d'images de difficulté croissante : MNIST → CIFAR-10 → Mammographies.

**Sommaire :**
1. **Partie 1 :** Classification MNIST (modèle linéaire + MLP + rétropropagation)
2. **Partie 2 :** Classification CIFAR-10 (architectures denses + CNN PyTorch)
3. **Partie 3 :** Diagnostic médical CBIS-DDSM (classification binaire mammographies)


In [ ]:
# Installation des dépendances (décommenter si nécessaire)
# !pip install torch torchvision matplotlib seaborn scikit-learn pillow tqdm

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Dataset
import torchvision
import torchvision.transforms as transforms
from PIL import Image
import os, time, warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.figsize': (10, 6), 'figure.dpi': 150,
                     'font.size': 11, 'axes.titlesize': 14, 'axes.titleweight': 'bold'})
PALETTE = ['#20808D', '#A84B2F', '#1B474D', '#BCE2E7', '#944454', '#FFC553']
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device utilisé : {DEVICE}")


---
# Partie 1 : Base de données MNIST

## 1.1 Le jeu de données

La base **MNIST** (Modified National Institute of Standards and Technology) contient 70 000 images de chiffres manuscrits en **28×28 pixels** (niveaux de gris).

- **60 000** images d'entraînement
- **10 000** images de test
- **10 classes** : chiffres 0 à 9

Chaque image $\vec{x}_i \in \mathbb{R}^{784}$ est un vecteur aplati dont les coordonnées correspondent aux intensités des pixels.


In [ ]:
# Chargement MNIST
transform_mnist = transforms.Compose([transforms.ToTensor()])
train_ds = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform_mnist)
test_ds = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform_mnist)

X_train_mnist = train_ds.data.numpy().astype(np.float32)
y_train_mnist = train_ds.targets.numpy()
X_test_mnist = test_ds.data.numpy().astype(np.float32)
y_test_mnist = test_ds.targets.numpy()

# Normalisation [0, 1] et aplatissement 28×28 → 784
X_train_flat = X_train_mnist.reshape(-1, 784) / 255.0
X_test_flat = X_test_mnist.reshape(-1, 784) / 255.0

print(f"Train: {X_train_flat.shape}, Test: {X_test_flat.shape}")
print(f"Valeurs normalisées: min={X_train_flat.min():.2f}, max={X_train_flat.max():.2f}")


In [ ]:
# Visualisation d'exemples MNIST
fig, axes = plt.subplots(2, 10, figsize=(15, 3.5))
fig.suptitle("Exemples de chiffres manuscrits MNIST", fontweight='bold')
for i in range(10):
    for j in range(2):
        idx = np.where(y_train_mnist == i)[0][j]
        axes[j, i].imshow(X_train_mnist[idx], cmap='gray')
        axes[j, i].axis('off')
        if j == 0: axes[j, i].set_title(str(i), fontsize=12)
plt.tight_layout()
plt.show()


In [ ]:
# Encodage one-hot des étiquettes
# y_i^(k) = 1 si l'image i appartient à la classe k, 0 sinon
def one_hot_encode(y, num_classes=10):
    Y = np.zeros((len(y), num_classes))
    Y[np.arange(len(y)), y] = 1.0
    return Y

Y_train_oh = one_hot_encode(y_train_mnist)
Y_test_oh = one_hot_encode(y_test_mnist)
print(f"Exemple: chiffre {y_train_mnist[0]} → one-hot: {Y_train_oh[0]}")


## 1.2.1 Modèle linéaire multi-classe

### Architecture

Pour chaque entrée $\vec{x} = (x_j)_{j \in [\![1,784]\!]}$, le modèle calcule un score (logit) :

$$o_k = \sum_{j=1}^{784} a_{k,j} \, x_j + a_{k,0}, \quad k \in \{0, \ldots, 9\}$$

Sous forme matricielle : $\vec{o} = A \vec{x} + \vec{b}$, avec $A \in \mathbb{R}^{10 \times 784}$, $\vec{b} \in \mathbb{R}^{10}$.

### Softmax

Les scores sont transformés en probabilités :

$$P_k(\vec{x}) = \frac{e^{o_k}}{\sum_{j=0}^{9} e^{o_j}}$$

### Prédiction

$$\hat{y} = \arg\max_k \, P_k(\vec{x})$$

### Fonction de coût (Cross-Entropy / Log Loss)

$$\mathfrak{L}(y, P) = -\frac{1}{n} \sum_{i=1}^{n} \sum_{k=0}^{9} y_i^{(k)} \cdot \ln(P_k(\vec{x}_i))$$

### Calcul du gradient

Le gradient de $\mathfrak{L}$ par rapport aux scores $\vec{o}$ vaut :

$$\frac{\partial \mathfrak{L}}{\partial o_k} = P_k - y^{(k)}$$

D'où les gradients par rapport aux paramètres :

$$\frac{\partial \mathfrak{L}}{\partial A} = \frac{1}{n} (P - Y)^T X, \qquad \frac{\partial \mathfrak{L}}{\partial b} = \frac{1}{n} \sum_i (P_i - Y_i)$$


In [ ]:
# Fonctions mathématiques fondamentales

def softmax(o):
    """Softmax avec stabilisation numérique"""
    o_shifted = o - np.max(o, axis=1, keepdims=True)
    exp_o = np.exp(o_shifted)
    return exp_o / np.sum(exp_o, axis=1, keepdims=True)

def cross_entropy_loss(Y_true, P_pred, eps=1e-12):
    """Entropie croisée (Log Loss)"""
    n = Y_true.shape[0]
    return -np.sum(Y_true * np.log(np.clip(P_pred, eps, 1 - eps))) / n

def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

def relu(x): return np.maximum(0, x)
def relu_derivative(x): return (x > 0).astype(float)


In [ ]:
class LinearClassifier:
    """Modèle linéaire multi-classe : x → Ax+b → softmax → P"""
    
    def __init__(self, d=784, K=10):
        self.A = np.random.randn(K, d) * np.sqrt(2.0 / d)  # Init. He
        self.b = np.zeros(K)
        self.history = {'loss_tr': [], 'loss_te': [], 'acc_tr': [], 'acc_te': []}
    
    def forward(self, X):
        return softmax(X @ self.A.T + self.b)
    
    def predict(self, X):
        return np.argmax(self.forward(X), axis=1)
    
    def train(self, Xtr, Ytr, ytr, Xte, Yte, yte, lr=0.5, epochs=30, bs=512):
        n = Xtr.shape[0]
        for ep in range(epochs):
            idx = np.random.permutation(n)
            for s in range(0, n, bs):
                e = min(s + bs, n)
                Xb, Yb = Xtr[idx[s:e]], Ytr[idx[s:e]]
                Pb = self.forward(Xb)
                delta = Pb - Yb  # ∂L/∂o = P - Y
                # Mise à jour : A -= lr * (1/n) * (P-Y)^T * X
                self.A -= lr * (delta.T @ Xb) / (e - s)
                self.b -= lr * np.mean(delta, axis=0)
            
            Ptr = self.forward(Xtr); Pte = self.forward(Xte)
            self.history['loss_tr'].append(cross_entropy_loss(Ytr, Ptr))
            self.history['loss_te'].append(cross_entropy_loss(Yte, Pte))
            self.history['acc_tr'].append(accuracy(ytr, np.argmax(Ptr, 1)))
            self.history['acc_te'].append(accuracy(yte, np.argmax(Pte, 1)))
            
            if (ep + 1) % 10 == 0:
                print(f"  Epoch {ep+1:3d} | Loss: {self.history['loss_tr'][-1]:.4f}/{self.history['loss_te'][-1]:.4f} | "
                      f"Acc: {self.history['acc_tr'][-1]*100:.2f}%/{self.history['acc_te'][-1]*100:.2f}%")

# Entraînement
print("Entraînement du modèle linéaire (30 époques, lr=0.5, batch=512)...")
model_lin = LinearClassifier(784, 10)
model_lin.train(X_train_flat, Y_train_oh, y_train_mnist, X_test_flat, Y_test_oh, y_test_mnist,
                lr=0.5, epochs=30, bs=512)

err_lin = 1 - model_lin.history['acc_te'][-1]
print(f"\nTaux d'erreur test : {err_lin*100:.2f}%")
print(f"Nombre de paramètres : {784*10 + 10} = 7 850")


## 1.2.2 Modèle avec couches cachées (MLP + Rétropropagation)

### Architecture à H couches cachées

$$z_q^1 = \phi_1\left(\sum_{k=1}^{784} a_{qk}^1 x_k + a_q^1\right), \qquad z_q^2 = \phi_2\left(\sum_{k=1}^{p_1} a_{qk}^2 z_k^1 + a_q^2\right)$$

$$o_q = \sum_{k=1}^{p_H} a_{qk}^{H+1} z_k^H + a_q^{H+1}, \quad q \in \{0, \ldots, 9\}$$

### Rétropropagation (Backpropagation)

1. **Dernière couche :** $\delta^{(H+1)} = P - Y$
2. **Couches cachées (en remontant) :** $\delta^{(h)} = (\delta^{(h+1)} W^{(h+1)}) \odot \phi'(o^{(h)})$
3. **Mise à jour :** $W^{(h)} \leftarrow W^{(h)} - \eta \cdot \frac{1}{n} (\delta^{(h)})^T z^{(h-1)}$

**Fonction d'activation choisie :** ReLU $\phi(x) = \max(0, x)$

**Fonction de coût :** Cross-Entropy (Log Loss)


In [ ]:
class MLPClassifier:
    """Perceptron multi-couches avec rétropropagation"""
    
    def __init__(self, dims):
        """dims = [784, p1, p2, ..., 10]"""
        self.L = len(dims) - 1
        self.W = [np.random.randn(dims[i+1], dims[i]) * np.sqrt(2.0/dims[i]) for i in range(self.L)]
        self.b = [np.zeros(dims[i+1]) for i in range(self.L)]
        self.history = {'loss_tr': [], 'loss_te': [], 'acc_tr': [], 'acc_te': []}
    
    def forward(self, X):
        self.z = [X]  # Stockage des activations pour la backprop
        self.o = []   # Pré-activations
        cur = X
        for i in range(self.L):
            oi = cur @ self.W[i].T + self.b[i]
            self.o.append(oi)
            cur = relu(oi) if i < self.L - 1 else softmax(oi)
            self.z.append(cur)
        return cur
    
    def backward(self, Y, lr):
        """
        Rétropropagation :
        δ^(last) = P - Y                    (softmax + cross-entropy)
        δ^(h) = (δ^(h+1) × W^(h+1)) ⊙ φ'(o^h)  (chaîne de dérivation)
        W^h -= lr/n × δ^h^T × z^(h-1)      (mise à jour des poids)
        b^h -= lr/n × Σ δ^h                 (mise à jour des biais)
        """
        n = Y.shape[0]
        delta = self.z[-1] - Y  # Gradient dernière couche
        
        for i in range(self.L - 1, -1, -1):
            dW = (delta.T @ self.z[i]) / n
            db = np.mean(delta, axis=0)
            if i > 0:
                delta = (delta @ self.W[i]) * relu_derivative(self.o[i-1])
            self.W[i] -= lr * dW
            self.b[i] -= lr * db
    
    def predict(self, X):
        return np.argmax(self.forward(X), axis=1)
    
    def count_params(self):
        return sum(w.size + b.size for w, b in zip(self.W, self.b))
    
    def train(self, Xtr, Ytr, ytr, Xte, Yte, yte, lr=0.1, epochs=30, bs=512):
        n = Xtr.shape[0]
        for ep in range(epochs):
            idx = np.random.permutation(n)
            for s in range(0, n, bs):
                e = min(s + bs, n)
                self.forward(Xtr[idx[s:e]])
                self.backward(Ytr[idx[s:e]], lr)
            Ptr = self.forward(Xtr); Pte = self.forward(Xte)
            self.history['loss_tr'].append(cross_entropy_loss(Ytr, Ptr))
            self.history['loss_te'].append(cross_entropy_loss(Yte, Pte))
            self.history['acc_tr'].append(accuracy(ytr, np.argmax(Ptr, 1)))
            self.history['acc_te'].append(accuracy(yte, np.argmax(Pte, 1)))
            if (ep + 1) % 10 == 0:
                print(f"  Ep {ep+1:3d} | Acc: {self.history['acc_tr'][-1]*100:.2f}%/{self.history['acc_te'][-1]*100:.2f}%")


In [ ]:
# --- MLP 1 couche cachée (H=1, p1=128) ---
print(">> Architecture : 784 → 128 (ReLU) → 10 (softmax)")
m1 = MLPClassifier([784, 128, 10])
m1.train(X_train_flat, Y_train_oh, y_train_mnist, X_test_flat, Y_test_oh, y_test_mnist, lr=0.1, epochs=30, bs=512)
err_1l = 1 - m1.history['acc_te'][-1]
print(f"Err test: {err_1l*100:.2f}% | Params: {m1.count_params()}")

# --- MLP 2 couches cachées (H=2, p1=128, p2=64) ---
print("\n>> Architecture : 784 → 128 (ReLU) → 64 (ReLU) → 10 (softmax)")
m2 = MLPClassifier([784, 128, 64, 10])
m2.train(X_train_flat, Y_train_oh, y_train_mnist, X_test_flat, Y_test_oh, y_test_mnist, lr=0.1, epochs=30, bs=512)
err_2l = 1 - m2.history['acc_te'][-1]
print(f"Err test: {err_2l*100:.2f}% | Params: {m2.count_params()}")


In [ ]:
# Courbes comparatives
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Comparaison des modèles MNIST", fontweight='bold')
for idx, (name, mdl) in enumerate([("Linéaire", model_lin), ("1 couche (128)", m1), ("2 couches (128,64)", m2)]):
    axes[idx].plot([a*100 for a in mdl.history['acc_tr']], label='Train', color=PALETTE[0], lw=2)
    axes[idx].plot([a*100 for a in mdl.history['acc_te']], label='Test', color=PALETTE[1], lw=2)
    axes[idx].set_xlabel('Époque'); axes[idx].set_ylabel('Précision (%)')
    axes[idx].set_title(name); axes[idx].legend(); axes[idx].set_ylim([85, 100])
plt.tight_layout(); plt.show()

# Tableau récapitulatif
print("\n" + "="*60)
print(f"{'Modèle':<25} {'Params':>8} {'Err. Train':>12} {'Err. Test':>12}")
print("-"*60)
print(f"{'Linéaire':<25} {7850:>8} {(1-model_lin.history['acc_tr'][-1])*100:>11.2f}% {err_lin*100:>11.2f}%")
print(f"{'1 couche (128)':<25} {m1.count_params():>8} {(1-m1.history['acc_tr'][-1])*100:>11.2f}% {err_1l*100:>11.2f}%")
print(f"{'2 couches (128,64)':<25} {m2.count_params():>8} {(1-m2.history['acc_tr'][-1])*100:>11.2f}% {err_2l*100:>11.2f}%")
print("="*60)


## 1.2.3 Analyse des erreurs et visualisation 2D

In [ ]:
# Matrice de confusion
best_pred = m2.predict(X_test_flat)
cm = confusion_matrix(y_test_mnist, best_pred)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, xticklabels=range(10), yticklabels=range(10))
ax.set_xlabel('Prédiction'); ax.set_ylabel('Vérité')
ax.set_title('Matrice de confusion - MLP 2 couches (MNIST)', fontweight='bold')
plt.tight_layout(); plt.show()

# Chiffres mal classés
mis = np.where(best_pred != y_test_mnist)[0]
print(f"Nombre de chiffres mal classés : {len(mis)} / {len(y_test_mnist)}")

fig, axes = plt.subplots(2, 10, figsize=(15, 4))
fig.suptitle("Exemples de chiffres mal classés (Vérité → Prédiction)", fontweight='bold')
for i in range(min(20, len(mis))):
    ax = axes[i//10, i%10]; ax.imshow(X_test_mnist[mis[i]], cmap='gray')
    ax.set_title(f"V:{y_test_mnist[mis[i]]}→P:{best_pred[mis[i]]}", fontsize=8, color='red')
    ax.axis('off')
plt.tight_layout(); plt.show()


In [ ]:
# Visualisation PCA 2D
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_test_flat)

fig, ax = plt.subplots(figsize=(10, 8))
sc = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=y_test_mnist, cmap='tab10', s=3, alpha=0.4)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title('Projection PCA 2D des données MNIST', fontweight='bold')
plt.colorbar(sc, ax=ax, label='Classe')
plt.tight_layout(); plt.show()


### Discussion sur les erreurs MNIST

- **Confusions fréquentes :** 4↔9, 3↔5, 7↔1, 2↔7
- **Chiffres ambigus :** écritures atypiques, traits non fermés, similitudes visuelles
- **PCA :** les classes proches (4/9, 3/5) se chevauchent dans l'espace des composantes principales
- **Modèle linéaire :** ne peut capturer que des frontières linéaires → erreur ~8%
- **Couches cachées + ReLU :** introduisent des non-linéarités → erreur ~3%
- **MNIST reste un problème simple** : fond uniforme noir, chiffres centrés, pas de rotation ni de bruit


---
# Partie 2 : Base de données CIFAR-10

## 2.1 Le jeu de données

**CIFAR-10** : 60 000 images couleur 32×32×3 (RGB), 10 classes :
{avion, automobile, oiseau, chat, cerf, chien, grenouille, cheval, bateau, camion}

- 50 000 images d'entraînement, 10 000 images de test
- Bien plus complexe que MNIST : objets variés, bruit de fond, variabilité de pose/position


In [ ]:
# Chargement CIFAR-10
train_ds_c = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
test_ds_c = torchvision.datasets.CIFAR10(root='./data', train=False, download=True)
X_tr_c = np.array(train_ds_c.data, dtype=np.float32)
y_tr_c = np.array(train_ds_c.targets)
X_te_c = np.array(test_ds_c.data, dtype=np.float32)
y_te_c = np.array(test_ds_c.targets)
CLASSES = ['avion','auto','oiseau','chat','cerf','chien','grenouille','cheval','bateau','camion']

fig, axes = plt.subplots(2, 10, figsize=(15, 4))
fig.suptitle("Exemples CIFAR-10", fontweight='bold')
for i in range(10):
    for j in range(2):
        idx = np.where(y_tr_c == i)[0][j]
        axes[j, i].imshow(X_tr_c[idx].astype(np.uint8)); axes[j, i].axis('off')
        if j == 0: axes[j, i].set_title(CLASSES[i], fontsize=8)
plt.tight_layout(); plt.show()


## 2.2 Travail préliminaire : architectures denses sur CIFAR-10

### Conversion en niveaux de gris
$$x_j = 0.299 R_j + 0.587 G_j + 0.114 B_j$$

On teste d'abord les architectures MNIST (linéaire et MLP) sur CIFAR-10, en niveaux de gris puis en couleur.


In [ ]:
Ytr_c = one_hot_encode(y_tr_c); Yte_c = one_hot_encode(y_te_c)

# Niveaux de gris
Xtr_g = (0.299*X_tr_c[:,:,:,0] + 0.587*X_tr_c[:,:,:,1] + 0.114*X_tr_c[:,:,:,2]).reshape(-1,1024)/255.0
Xte_g = (0.299*X_te_c[:,:,:,0] + 0.587*X_te_c[:,:,:,1] + 0.114*X_te_c[:,:,:,2]).reshape(-1,1024)/255.0

# Couleur (vecteur 3072 = 32×32×3)
Xtr_col = X_tr_c.reshape(-1, 3072) / 255.0
Xte_col = X_te_c.reshape(-1, 3072) / 255.0

# Sous-ensemble pour accélérer l'entraînement numpy
sub_tr, sub_te = 10000, 5000
idx_tr = np.random.choice(len(Xtr_g), sub_tr, replace=False)
idx_te = np.random.choice(len(Xte_g), sub_te, replace=False)

print(">> Linéaire (niveaux de gris)")
m_cg = LinearClassifier(1024, 10)
m_cg.train(Xtr_g[idx_tr], Ytr_c[idx_tr], y_tr_c[idx_tr], Xte_g[idx_te], Yte_c[idx_te], y_te_c[idx_te], lr=0.1, epochs=20, bs=512)
print(f"   Err test: {(1-m_cg.history['acc_te'][-1])*100:.2f}%")

print("\n>> MLP 2 couches (niveaux de gris)")
m_mg = MLPClassifier([1024, 256, 128, 10])
m_mg.train(Xtr_g[idx_tr], Ytr_c[idx_tr], y_tr_c[idx_tr], Xte_g[idx_te], Yte_c[idx_te], y_te_c[idx_te], lr=0.05, epochs=20, bs=512)
print(f"   Err test: {(1-m_mg.history['acc_te'][-1])*100:.2f}%")

print("\n>> Linéaire (couleur RGB)")
m_cc = LinearClassifier(3072, 10)
m_cc.train(Xtr_col[idx_tr], Ytr_c[idx_tr], y_tr_c[idx_tr], Xte_col[idx_te], Yte_c[idx_te], y_te_c[idx_te], lr=0.05, epochs=20, bs=512)
print(f"   Err test: {(1-m_cc.history['acc_te'][-1])*100:.2f}%")

print("\n>> MLP 2 couches (couleur RGB)")
m_mc = MLPClassifier([3072, 256, 128, 10])
m_mc.train(Xtr_col[idx_tr], Ytr_c[idx_tr], y_tr_c[idx_tr], Xte_col[idx_te], Yte_c[idx_te], y_te_c[idx_te], lr=0.05, epochs=20, bs=512)
print(f"   Err test: {(1-m_mc.history['acc_te'][-1])*100:.2f}%")


## 2.3 Étude des filtres de convolution

### Principe de la convolution

On fait glisser un noyau (kernel) $K$ de taille $3 \times 3$ sur l'image avec zero-padding :

$$m'_{u,v} = \sum_{u'=1}^{3} \sum_{v'=1}^{3} K_{u',v'} \, m_{u'+u-2,\, v'+v-2} + l$$

**6 filtres à étudier :**
- $K_1$ : Moyenne (flou)
- $K_2$ : Netteté (accentuation des contours)
- $K_3$ : Détection de bords verticaux
- $K_4$ : Gradient horizontal (Prewitt)
- $K_5$ : Gradient horizontal (Sobel)
- $K_6$ : Effet d'embossage diagonal


In [ ]:
# Implémentation de la convolution 2D
def conv2d_manual(image, kernel, bias=0):
    """Convolution 2D avec zero-padding"""
    h, w = image.shape
    kh, kw = kernel.shape
    ph, pw = kh // 2, kw // 2
    padded = np.pad(image, ((ph, ph), (pw, pw)), mode='constant', constant_values=0)
    output = np.zeros_like(image)
    for u in range(h):
        for v in range(w):
            output[u, v] = np.sum(padded[u:u+kh, v:v+kw] * kernel) + bias
    return output

# Définition des 6 filtres
K1 = np.ones((3,3)) / 9.0                               # Moyenne
K2 = np.array([[0,-1,0],[-1,5,-1],[0,-1,0]])            # Netteté
K3 = np.array([[-1,2,-1],[-1,2,-1],[-1,2,-1]])          # Bords verticaux
K4 = np.array([[-1,0,1],[-1,0,1],[-1,0,1]])             # Prewitt
K5 = np.array([[-1,0,1],[-2,0,2],[-1,0,1]])             # Sobel
K6 = np.array([[-2,-1,0],[-1,1,1],[0,1,2]])             # Emboss

# Image de test (chat CIFAR en niveaux de gris)
cat_idx = np.where(y_tr_c == 3)[0][0]
sample = 0.299*X_tr_c[cat_idx,:,:,0] + 0.587*X_tr_c[cat_idx,:,:,1] + 0.114*X_tr_c[cat_idx,:,:,2]

filters = [K1, K2, K3, K4, K5, K6]
fnames = ['K1: Moyenne (flou)', 'K2: Netteté', 'K3: Bords verticaux',
          'K4: Prewitt horiz.', 'K5: Sobel horiz.', 'K6: Emboss diag.']

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle("Effet des filtres de convolution (l=0)", fontweight='bold')
axes[0, 0].imshow(sample, cmap='gray'); axes[0, 0].set_title('Originale'); axes[0, 0].axis('off')
for i, (K, nm) in enumerate(zip(filters, fnames)):
    r, c = (i+1)//4, (i+1)%4
    axes[r, c].imshow(conv2d_manual(sample, K), cmap='gray'); axes[r, c].set_title(nm, fontsize=9); axes[r, c].axis('off')
axes[1, 3].axis('off')
plt.tight_layout(); plt.show()

print("Analyse des filtres :")
print("• K1 (Moyenne) : lisse l'image, réduit le bruit → flou")
print("• K2 (Netteté) : accentue les contours et détails")
print("• K3 (Bords verticaux) : détecte les lignes verticales")
print("• K4 (Prewitt) : gradient horizontal → contours verticaux")
print("• K5 (Sobel) : similaire à Prewitt, pondéré → plus sensible")
print("• K6 (Emboss) : effet de relief, détecte les diagonales")


## 2.5/2.6 Réseau de neurones convolutif (CNN) avec PyTorch

### Architecture (selon l'énoncé)

| Couche | Opération | Sortie |
|--------|-----------|--------|
| Entrée | Image couleur | 32×32×3 |
| Conv1 | 64 filtres 3×3×3, padding=1 | 32×32×64 |
| Conv2 | 64 filtres 3D 3×3×64, padding=1 | 32×32×64 |
| Pool1 | MaxPool 2×2 | 16×16×64 |
| Conv3 | 64 filtres 3D 3×3×64, padding=1 | 16×16×64 |
| Pool2 | MaxPool 2×2 | 8×8×64 |
| Conv4 | 64 filtres 3D 3×3×64, padding=1 | 8×8×64 |
| Flatten | Aplatissement | 4096 |
| FC1 | Dense 4096→256 | 256 |
| FC2 | Dense 256→10 + Softmax | 10 |

### Convolution sur des volumes (filtres 3D)

$$m'_{u,v} = \sum_{c=1}^{C} \left( \sum_{u'=1}^{3} \sum_{v'=1}^{3} K_{u',v',c} \cdot m_{u+u'-2,\,v+v'-2,\,c} \right) + l$$

### Max-Pooling

$$m''_{u,v} = \max_{(u',v') \in \{0,1\}^2} m'_{2u+u'-1,\, 2v+v'-1}$$

### Rétropropagation dans le CNN (Option B : PyTorch)
On utilise **PyTorch** pour le calcul automatique des gradients (Autograd).


In [ ]:
class CIFAR10_CNN(nn.Module):
    """
    CNN pour CIFAR-10 selon l'architecture de l'énoncé.
    Conv64 → Conv64 → MaxPool → Conv64 → MaxPool → Conv64 → Flatten → FC(256) → FC(10)
    """
    def __init__(self):
        super().__init__()
        # Couche 1 : 64 filtres couleur (3→64)
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(64)
        # Couche 2 : 64 filtres 3D (64→64)
        self.conv2 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2)  # 32→16
        # Couche 4 : 64 filtres 3D
        self.conv3 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2)  # 16→8
        # Couche 6 : 64 filtres 3D
        self.conv4 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(64)
        self.dropout = nn.Dropout(0.25)
        # Densification : 8×8×64 = 4096 → 256 → 10
        self.fc1 = nn.Linear(8*8*64, 256)
        self.fc2 = nn.Linear(256, 10)
    
    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))   # 32×32×64
        x = F.relu(self.bn2(self.conv2(x)))   # 32×32×64
        x = self.dropout(self.pool1(x))        # 16×16×64
        x = F.relu(self.bn3(self.conv3(x)))   # 16×16×64
        x = self.dropout(self.pool2(x))        # 8×8×64
        x = F.relu(self.bn4(self.conv4(x)))   # 8×8×64
        x = torch.flatten(x, 1)                # 4096
        x = F.relu(self.fc1(x))               # 256
        return self.fc2(self.dropout(x))        # 10

# Préparation données avec augmentation
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])
transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])

trainset = torchvision.datasets.CIFAR10('./data', True, transform=transform_train)
testset = torchvision.datasets.CIFAR10('./data', False, transform=transform_test)
trainloader = DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)
testloader = DataLoader(testset, batch_size=128, shuffle=False, num_workers=2)

# Entraînement
cnn = CIFAR10_CNN().to(DEVICE)
print(f"Paramètres CNN : {sum(p.numel() for p in cnn.parameters() if p.requires_grad):,}")
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(cnn.parameters(), lr=0.001, weight_decay=1e-4)

# Note: sur GPU, entraîner 20-30 époques. Sur CPU, réduire ou utiliser Google Colab.
EPOCHS = 15  # Ajuster selon le hardware
for epoch in range(EPOCHS):
    cnn.train()
    running_loss, correct, total = 0., 0, 0
    for inputs, labels in trainloader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = cnn(inputs)
        loss = criterion(outputs, labels)
        loss.backward()  # Rétropropagation automatique (Autograd)
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    if (epoch + 1) % 5 == 0:
        print(f"  Epoch {epoch+1}/{EPOCHS} | Loss: {running_loss/total:.4f} | Acc train: {correct/total*100:.2f}%")

# Évaluation
cnn.eval()
correct, total = 0, 0
all_preds, all_labels = [], []
with torch.no_grad():
    for inputs, labels in testloader:
        inputs = inputs.to(DEVICE)
        outputs = cnn(inputs)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

print(f"\nTaux d'erreur CNN test : {(1 - correct/total)*100:.2f}%")


---
# Partie 3 : Application au diagnostic médical (CBIS-DDSM)

## Objectif
Classification binaire de mammographies : **bénin** vs **malin** (cancer)

### Le dataset CBIS-DDSM
- Images de mammographies haute résolution (~4000×3000 pixels)
- Étiquettes dans  (colonne )
- Regroupement : BENIGN + BENIGN_WITHOUT_CALLBACK → 0 (bénin), MALIGNANT → 1 (malin)

### Défis spécifiques
- **Déséquilibre des classes** : plus de cas bénins que malins → pondération
- **Redimensionnement** : 128×128 ou 224×224 pour tenir en mémoire
- **Importance des faux négatifs** : un cancer manqué = retard de traitement


In [ ]:
class MammographyCNN(nn.Module):
    """
    CNN pour classification binaire de mammographies.
    Entrée : images niveaux de gris 128×128
    Sortie : 2 classes (bénin / malin)
    """
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            # Bloc 1 : extraction bas niveau
            nn.Conv2d(1, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.Conv2d(32, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(),
            nn.MaxPool2d(2, 2), nn.Dropout(0.25),  # 128→64
            # Bloc 2 : niveau intermédiaire
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.MaxPool2d(2, 2), nn.Dropout(0.25),  # 64→32
            # Bloc 3 : haut niveau
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(),
            nn.MaxPool2d(2, 2), nn.Dropout(0.25),  # 32→16
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((4, 4)),
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, 2)
        )
    
    def forward(self, x):
        return self.classifier(self.features(x))

# Fonction de chargement CBIS-DDSM (à adapter selon votre arborescence)
def load_cbis_ddsm(csv_path, image_root):
    """
    Charge le dataset CBIS-DDSM depuis le CSV et les images.
    
    Args:
        csv_path: chemin vers mass_case_description_train_set.csv
        image_root: dossier racine des images
    
    Returns:
        image_paths, labels (0=bénin, 1=malin)
    """
    import csv
    paths, labels = [], []
    with open(csv_path) as f:
        for row in csv.DictReader(f):
            pathology = row['pathology'].strip()
            img_path = os.path.join(image_root, row['image file path'].strip())
            if os.path.exists(img_path):
                if pathology in ['BENIGN', 'BENIGN_WITHOUT_CALLBACK']:
                    labels.append(0)
                elif pathology == 'MALIGNANT':
                    labels.append(1)
                else:
                    continue
                paths.append(img_path)
    return paths, np.array(labels)

# Entraînement (avec données réelles)
def train_mammography(train_loader, test_loader, class_weights, epochs=30):
    model = MammographyCNN().to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=torch.FloatTensor(class_weights).to(DEVICE))
    optimizer = optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-4)
    
    for epoch in range(epochs):
        model.train()
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(inputs), labels)
            loss.backward()
            optimizer.step()
        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1}/{epochs}")
    
    return model

mammo = MammographyCNN()
print(f"Paramètres MammoCNN : {sum(p.numel() for p in mammo.parameters() if p.requires_grad):,}")
print("\nPour l'entraînement réel : télécharger CBIS-DDSM depuis")
print("https://wiki.cancerimagingarchive.net/pages/viewpage.action?pageId=22516629")


### Analyse de la matrice de confusion en diagnostic médical

| | Prédit Bénin | Prédit Malin |
|---|---|---|
| **Réel Bénin** | Vrai Négatif (TN) | Faux Positif (FP) |
| **Réel Malin** | **Faux Négatif (FN)** | Vrai Positif (TP) |

- **Faux Négatifs (FN)** : cancers **manqués** → conséquence la plus grave (retard de diagnostic)
- **Faux Positifs (FP)** : fausses alarmes → biopsies inutiles mais non dangereuses
- En diagnostic médical, on privilégie la **sensibilité (rappel)** = TP / (TP + FN)

$$\text{Sensibilité} = \frac{\text{Vrais Positifs}}{\text{Vrais Positifs + Faux Négatifs}}$$

Même au prix de plus de faux positifs, il faut maximiser la détection des vrais cancers.


---
# Synthèse finale

| Dataset | Modèle | Taux d'erreur test |
|---------|--------|-------------------|
| MNIST | Linéaire | ~8% |
| MNIST | MLP 1 couche (128) | ~3-4% |
| MNIST | MLP 2 couches (128, 64) | ~3% |
| CIFAR-10 | Linéaire (gris) | ~75% |
| CIFAR-10 | MLP 2 couches (gris) | ~70% |
| CIFAR-10 | MLP 2 couches (couleur) | ~65% |
| CIFAR-10 | **CNN** | ~**20-25%** (15-20 epochs) |

### Conclusions

1. **Les couches cachées** améliorent les performances en introduisant des non-linéarités
2. **La convolution** est essentielle pour les images naturelles : elle capture les motifs locaux (bords, textures) tout en étant invariante par translation
3. **Le max-pooling** réduit la dimensionnalité et apporte une robustesse aux petites translations
4. **CIFAR-10 vs MNIST** : les architectures denses échouent sur CIFAR car les images naturelles ont un fond complexe, des objets de taille/position variables
5. **Le diagnostic médical** ajoute les défis du déséquilibre de classes et l'importance critique des métriques (sensibilité > précision)
